In [1]:
import os
import requests
from dotenv import load_dotenv
from neo4j import GraphDatabase, basic_auth
from typing import List
from neo4j_graphrag.embeddings.base import Embedder
from neo4j_graphrag.retrievers import VectorRetriever


load_dotenv()

driver = GraphDatabase.driver(
  os.getenv('neo4j_url'),
  auth=basic_auth("neo4j", os.getenv('neo4j_name')))


class HyperClovaXEmbeddings(Embedder):
    def __init__(self):
        self.url = "https://clovastudio.stream.ntruss.com/testapp/v1/api-tools/embedding/v2"
        api_key = os.getenv('CLOVA_API_KEY')
        if not api_key:
            raise ValueError("CLOVA_API_KEY 환경 변수가 설정되지 않았습니다.")
        
        self.headers = {
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        }

    def embed_query(self, text: str) -> List[float]:
        data = {"text": text}
        try:
            response = requests.post(self.url, headers=self.headers, json=data)
            response.raise_for_status() 
            result = response.json()
            
            if result and "result" in result and "embedding" in result["result"]:
                return result["result"]["embedding"]
            else:
                print(f"경고: API 응답에서 임베딩을 찾을 수 없습니다. 응답: {result}")
                return []
        except requests.exceptions.RequestException as e:
            print(f"API 요청 중 오류 발생: {e}")
            return []

embedder = HyperClovaXEmbeddings()

retriever = VectorRetriever(
    driver,
    index_name='moviePlotsEmbedding',
    embedder=embedder,
    return_properties=['title', 'plot']
)


In [47]:
import os
import requests
from typing import List, Optional, Dict, Any
from neo4j_graphrag.generation import GraphRAG
from types import SimpleNamespace

class HyperClovaXLLM:
    def __init__(
        self,
        model_name: str = "HCX-003",
        api_key: Optional[str] = None,
        base_url: str = "https://clovastudio.stream.ntruss.com/testapp/v1",
        model_params: Optional[Dict[str, Any]] = None,
    ):
        self.model_name = model_name
        self.api_key = api_key or os.getenv("CLOVA_API_KEY")
        if not self.api_key:
            raise ValueError("CLOVA_API_KEY 환경변수가 필요합니다.")
        self.base_url = base_url
        self.model_params = model_params or {}

    @property
    def headers(self):
        return {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

    def get_messages(
        self,
        input: str,
        message_history: Optional[List[Dict[str, str]]] = None,
        system_instruction: Optional[str] = None,
    ) -> List[Dict[str, str]]:
        messages = []
        if system_instruction:
            messages.append({"role": "system", "content": system_instruction})
        if message_history:
            messages.extend(message_history)
        messages.append({"role": "user", "content": input})
        return messages

    def invoke(
        self,
        input: str,
        message_history: Optional[List[Dict[str, str]]] = None,
        system_instruction: Optional[str] = None,
    ) -> str:
        url = f"{self.base_url}/chat-completions/{self.model_name}"
        data = {
            "messages": self.get_messages(input, message_history, system_instruction),
            "maxTokens": self.model_params.get("maxTokens", 1000),
            "temperature": self.model_params.get("temperature", 0.7),
        }
        response = requests.post(url, headers=self.headers, json=data)
        response.raise_for_status()
        result = response.json()
        # 응답 구조에 따라 content 추출

        if "result" in result and "message" in result["result"]:
            return SimpleNamespace(**result["result"]["message"])
        elif "choices" in result and len(result["choices"]) > 0:
            return SimpleNamespace(**result["choices"][0]["message"])
        else:
            return str(result)


In [48]:
llm = HyperClovaXLLM(model_name='HCX-003')

In [49]:
rag = GraphRAG(retriever=retriever, llm=llm)

In [50]:
query_text = 'What movies are sad romances?'
response = rag.search(query_text=query_text, retriever_config={'top_k': 5})
print(response.answer)

c:\Users\Jo\AppData\Local\Programs\Python311\Lib\site-packages\neo4j_graphrag\generation\graphrag.py:120: DeprecationWarning: The default value of 'return_context' will change from 'False' to 'True' in a future version.
  warnings.warn(
c:\Users\Jo\AppData\Local\Programs\Python311\Lib\site-packages\neo4j_graphrag\retrievers\vector.py:201: DeprecationWarning: The default returned 'id' field in the search results will be removed. Please switch to using 'elementId' instead.
  search_query, search_params = get_search_query(


Based on the provided context, the movie "Bed of Roses" can be considered a sad romance as it is a romantic drama about a young career girl who falls in love with a shy florist, but there isn't enough information to determine if the ending is happy or sad.


In [1]:
from neo4j import GraphDatabase, basic_auth
import openai

driver = GraphDatabase.driver(
  "neo4j://52.4.166.125:7687",
  auth=basic_auth("neo4j", "stage-alkalinity-crashes"))

In [2]:
import os
import requests
from dotenv import load_dotenv
import openai
import neo4j, neo4j_genai, neo4j_graphrag

load_dotenv()

True

In [3]:
def generate_embedding(text):
    embedding = openai.embeddings.create(input = [text], model='text-embedding-ada-002').data[0].embedding
    return embedding

In [5]:
from neo4j_genai.retrievers import VectorRetriever
from neo4j_genai.embeddings.openai import OpenAIEmbeddings
embedder = OpenAIEmbeddings(model='text-embedding-ada-002')
retriever = VectorRetriever(
    driver,
    index_name='moviePlotsEmbedding',
    embedder=embedder,
    return_properties=['title', 'plot'],
)

In [6]:
query_text = 'A movie about a shooting incident.'
retriever_result = retriever.search(query_text=query_text, top_k=3)
print(retriever_result)

items=[RetrieverResultItem(content="{'title': 'City Hall', 'plot': 'The accidental shooting of a boy in New York leads to an investigation by the Deputy Mayor, and unexpectedly far-reaching consequences.'}", metadata={'score': 0.9315948486328125, 'nodeLabels': None, 'id': None}), RetrieverResultItem(content="{'title': 'Usual Suspects, The', 'plot': 'A sole survivor tells of the twisty events leading up to a horrific gun battle on a boat, which begin when five criminals meet at a seemingly random police lineup.'}", metadata={'score': 0.930877685546875, 'nodeLabels': None, 'id': None}), RetrieverResultItem(content="{'title': 'Nick of Time', 'plot': 'An unimpressive, every-day man is forced into a situation where he is told to kill a politician to save his kidnapped daughter.'}", metadata={'score': 0.913909912109375, 'nodeLabels': None, 'id': None})] metadata={'__retriever': 'VectorRetriever'}


In [7]:
from neo4j_genai.llm import OpenAILLM
from neo4j_genai.generation import GraphRAG

llm = OpenAILLM(model_name='gpt-4o', model_params={'temperature': 0})

In [8]:
rag = GraphRAG(retriever=retriever, llm=llm)

In [11]:
retriever.search(query_text = 'what movies are sad romances?', top_k=5).items

[RetrieverResultItem(content="{'title': 'Bed of Roses', 'plot': 'Romantic drama about a young career girl who is swept off her feet by a shy florist, who fell in love with her after one glimpse through a shadowy window.'}", metadata={'score': 0.912261962890625, 'nodeLabels': None, 'id': None}),
 RetrieverResultItem(content="{'title': 'Postman, The (Postino, Il)', 'plot': 'Simple Italian postman learns to love poetry while delivering mail to a famous poet; he uses this to woo local beauty Beatrice.'}", metadata={'score': 0.8889312744140625, 'nodeLabels': None, 'id': None}),
 RetrieverResultItem(content="{'title': 'Beautiful Girls', 'plot': 'A piano player at a crossroads in his life returns home to his friends and their own problems with life and love.'}", metadata={'score': 0.8872222900390625, 'nodeLabels': None, 'id': None}),
 RetrieverResultItem(content='{\'title\': \'American President, The\', \'plot\': "Comedy-drama about a widowed U.S. president and a lobbyist who fall in love. It

In [9]:
query_text = 'what movies are sad romances?'
response = rag.search(query_text=query_text, retriever_config={'top_k':5})
print(response.answer)

The movies "Bed of Roses" and "How to Make an American Quilt" could be considered sad romances. "Bed of Roses" is a romantic drama about a young career girl and a shy florist, which may involve emotional elements. "How to Make an American Quilt" involves tales of romance and sorrow, suggesting a mix of sad and romantic themes.


In [12]:
from neo4j_genai.retrievers import Text2CypherRetriever

llm = OpenAILLM(model_name='gpt-4o', model_params={'temperature': 0})

In [13]:
from neo4j import GraphDatabase
from neo4j.time import Date

def get_node_datatype(value):
    '''
    입력된 노드 Value의 데이터 타입을 반환하는 함수
    '''
    if isinstance(value, str):
        return 'STRING'
    elif isinstance(value, int):
        return 'INTEGER'
    elif isinstance(value, float):
        return 'FLOAT'
    elif isinstance(value, bool):
        return 'BOOLEAN'
    elif isinstance(value, list):
        return f'LIST[{get_node_datatype(value[0])}]' if value else "LIST"
    elif isinstance(value, Date):
        return 'DATE'
    else:
        return 'UNKNOWN'

In [ ]:
def get_schema(uri, user, password):
    '''
    Graph DB의 정보를 받아 노드 및 관계의 프로퍼티를 추출하고 스키마 딕셔너리를 반환하는 함수
    '''
    driver = GraphDatabase.driver(
        uri,
        auth=basic_auth(user, password)
    )

    with driver.session() as session:
        node_query = '''
        MATCH (n)
        WITH DISTINCT labels(n) AS node_labels, keys(n) AS property_keys, n
        UNWIND node_labels AS label
        UNWIND property_keys AS key
        RETURN label, key, n[key] AS sample_value
        '''
        nodes = session.run(node_query)

        rel_query = '''
        MATCH ()-[r]->()
        WITH DISTINCT type(r) AS rel_type, keys(r) AS property_keys, r
        UNWIND property_keys AS key
        RETURN rel_type, key, r[key] AS sample_value
        '''
        relationships = session.run(rel_query)

        